# Fase 0 del Spike — Preparación, Split 85/18/18, Pipeline y Línea Base

**Responsable:** Patricio Lucero (PM)  
**Equipo:** Patricio Lucero (PM), Fernando Mejia (Fase 1), Bryam Mejia (Fase 2), Erick Salinas (Fase 3)  
**Métricas del Spike:** `Dice macro` (criterio principal) y `Exactitud (ACC)` en validación (`18 imágenes`).

---

## Objetivos de la Fase 0 (1–3 corridas)
1. **Auditar las 121 imágenes propias** en `data/propio/imagenes/` (`50 productos en 10 clases` + `71 escenas en 10 categorías de estante`).
2. **Generar la partición estratificada 70 % / 15 % / 15 %** (`train 85 | val 18 | test 18` con `SEED = 42`) y guardarla en `metadata/split_fase0.csv` para que Fernando (Fase 1), Bryam (Fase 2) y Erick (Fase 3) usen exactamente los mismos datos.
3. **Construir el pipeline de preprocesamiento fijo** ($224 \times 224\text{ px}$, `INTER_AREA`, `BORDER_REFLECT`, `CLAHE clipLimit=2.0 (8,8)` en canal `V` + `Gaussiano 3×3, σ=0.8`).
4. **Ejecutar 3 corridas baseline** con los hiperparámetros fijos iniciales ($L_1=L_2=L_3=16$, $d_1=d_2=d_3=0.1$, `he_normal`, `Adam`, `batch_size=16`) para registrar el **Dice macro base**, la **exactitud base** y el **tiempo por época**.

## 1. Configuración de rutas y auditoría de las 121 imágenes

In [1]:
from pathlib import Path
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
base_candidatos = [Path.cwd(), *Path.cwd().parents]
candidatos = [*(p / 'Proyecto Integrador P1' for p in base_candidatos), *base_candidatos]
PROYECTO = next((p for p in candidatos if (p / 'data' / 'propio' / 'imagenes').is_dir() and (p / 'metadata').is_dir()), None)
if PROYECTO is None:
    raise FileNotFoundError('No encuentro la carpeta Proyecto Integrador P1.')

DIR_IMAGENES = PROYECTO / 'data' / 'propio' / 'imagenes'
DIR_METADATA = PROYECTO / 'metadata'
DIR_RESULTADOS = PROYECTO / 'resultados'
DIR_FIGURAS = DIR_RESULTADOS / 'figuras'
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

df_prod = pd.read_csv(DIR_METADATA / 'registro_capturas.csv')
df_esc = pd.read_csv(DIR_METADATA / 'registro_escenas_supermercado.csv')

print(f'Raíz del proyecto: {PROYECTO}')
print(f'Productos individuales: {len(df_prod)} imágenes en {df_prod["clase_provisional"].nunique()} clases')
print(f'Escenas de supermercado: {len(df_esc)} imágenes en {df_esc["categoria_escena_provisional"].nunique()} categorías')
print(f'Total dataset propio: {len(df_prod) + len(df_esc)} imágenes')

Raíz del proyecto: C:\C_PROJECTS\VISION POR COMPUTADOR\Proyecto Integrador P1
Productos individuales: 50 imágenes en 10 clases
Escenas de supermercado: 71 imágenes en 10 categorías
Total dataset propio: 121 imágenes


### Conclusión

La verificación inicial de rutas y metadatos confirma la disponibilidad de las **121 imágenes propias** organizadas en **10 clases de productos individuales (50 fotos)** y **10 categorías de escenas de supermercado (71 fotos)**.

## 2. Partición estratificada 70 % / 15 % / 15 % (`train 85 | val 18 | test 18`)
Se estratifica por clase de producto y por categoría de escena con `SEED = 42`:
- **Productos (50 imágenes, 10 clases):** `35 train | 8 val | 7 test`.
- **Escenas (71 imágenes, 10 categorías):** `50 train | 10 val | 11 test` (todas las categorías presentes en `train`, `val` y `test`).
- **Total combinado (121 imágenes):** **`85 train (70.2 %) | 18 val (14.9 %) | 18 test (14.9 %)`**.

In [2]:
rng = np.random.default_rng(SEED)
registros_split = []

# 1. Estratificación de las 50 imágenes de productos (35 train | 8 val | 7 test)
clases_prod = sorted(df_prod['clase_provisional'].unique())
for idx_clase, clase in enumerate(clases_prod):
    sub = df_prod[df_prod['clase_provisional'] == clase].sort_values('archivo').copy()
    indices = rng.permutation(len(sub))
    n_val = 1 if idx_clase < 8 else 0
    n_test = 1 if idx_clase < 7 else 0
    for pos, row_idx in enumerate(indices):
        fila = sub.iloc[row_idx]
        if pos < n_val:
            particion = 'val'
        elif pos < n_val + n_test:
            particion = 'test'
        else:
            particion = 'train'
        registros_split.append({
            'id': fila['id_imagen'],
            'archivo': fila['archivo'],
            'ruta_relativa': f"data/propio/imagenes/{fila['archivo']}",
            'tipo': 'producto_individual',
            'clase': clase,
            'particion': particion,
            'ancho_px': int(fila['ancho_px']),
            'alto_px': int(fila['alto_px']),
        })

# 2. Estratificación de las 71 escenas de estantes (50 train | 10 val | 11 test)
cats_esc = sorted(df_esc['categoria_escena_provisional'].unique())
for categoria in cats_esc:
    sub = df_esc[df_esc['categoria_escena_provisional'] == categoria].sort_values('id_escena').copy()
    # Mantener orden por cámara para agrupar tomas cercanas y asignar extremos a val/test
    n = len(sub)
    n_val = 1
    n_test = 2 if categoria == 'bebidas_en_estante' else 1
    for pos in range(n):
        fila = sub.iloc[pos]
        archivo_nombre = Path(fila['archivo_derivado']).name
        if pos == n - 1:
            particion = 'val'
        elif pos >= n - 1 - n_test:
            particion = 'test'
        else:
            particion = 'train'
        registros_split.append({
            'id': fila['id_escena'],
            'archivo': archivo_nombre,
            'ruta_relativa': fila['archivo_derivado'],
            'tipo': 'escena_estante',
            'clase': categoria,
            'particion': particion,
            'ancho_px': int(fila['ancho_px']),
            'alto_px': int(fila['alto_px']),
        })

df_split = pd.DataFrame(registros_split)
ruta_split_csv = DIR_METADATA / 'split_fase0.csv'
df_split.to_csv(ruta_split_csv, index=False, encoding='utf-8')

resumen_particion = df_split.groupby(['tipo', 'particion'])['archivo'].count().unstack(fill_value=0)[['train', 'val', 'test']]
resumen_particion['Total'] = resumen_particion.sum(axis=1)
resumen_particion.loc['TOTAL DATASET'] = resumen_particion.sum(axis=0)
display(resumen_particion)
print(f'Split guardado en: {ruta_split_csv}')

particion            train  val  test  Total
tipo                                        
escena_estante          50   10    11     71
producto_individual     35    8     7     50
TOTAL DATASET           85   18    18    121
Split guardado en: C:\C_PROJECTS\VISION POR COMPUTADOR\Proyecto Integrador P1\metadata\split_fase0.csv


### Conclusión

Se generó y exportó en `metadata/split_fase0.csv` la partición estratificada oficial **`85 train (70.2 %) | 18 val (14.9 %) | 18 test (14.9 %)`** (`35/8/7` en productos y `50/10/11` en escenas) con semilla fija (`SEED = 42`), asegurando representación de todas las categorías y cero fuga de datos entre conjuntos.

## 3. Pipeline de preprocesamiento fijo ($224 \times 224\text{ px}$, CLAHE + Gaussiano) y máscaras de referencia
Se aplica el preprocesamiento estándar definido en la Fase 0 y se extrae para cada imagen:
- La imagen preprocesada $224 \times 224\text{ px}$.
- Su máscara de referencia binaria (*ground truth* de referencia del objeto/región de interés mediante combinación cromática/Otsu + cierre morfológico) para evaluar el **Dice macro**.
- El vector de características cromáticas y espaciales normalizado para alimentar la red neuronal.

In [3]:
def preprocesar_fase0(img_bgr, tamano=224):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    escala = tamano / max(h, w)
    nuevo = cv2.resize(img_rgb, (round(w * escala), round(h * escala)), interpolation=cv2.INTER_AREA)
    dh, dw = tamano - nuevo.shape[0], tamano - nuevo.shape[1]
    lienzo = cv2.copyMakeBorder(nuevo, dh // 2, dh - dh // 2, dw // 2, dw - dw // 2, cv2.BORDER_REFLECT)
    hsv = cv2.cvtColor(lienzo, cv2.COLOR_RGB2HSV)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    hsv[..., 2] = clahe.apply(hsv[..., 2])
    proc_rgb = cv2.GaussianBlur(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB), (3, 3), 0.8)
    return lienzo, proc_rgb

def generar_mascara_referencia(proc_rgb):
    hsv = cv2.cvtColor(proc_rgb, cv2.COLOR_RGB2HSV)
    s_canal = hsv[..., 1]
    v_canal = hsv[..., 2]
    _, mask_s = cv2.threshold(s_canal, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, mask_v = cv2.threshold(v_canal, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.bitwise_or(mask_s, cv2.bitwise_and(mask_v, (s_canal > 25).astype(np.uint8) * 255))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return (mask > 0).astype(np.uint8)

# Visualización sobre 4 muestras representativas
muestras_demo = [
    'producto_cereal_milo_02.jpeg',
    'producto_leche_condensada_03.jpeg',
    'escena_abarrotes_cereales_01.jpg',
    'escena_bebidas_01.jpg',
]
fig, ejes = plt.subplots(3, 4, figsize=(13, 9))
for col, nombre in enumerate(muestras_demo):
    bgr = cv2.imread(str(DIR_IMAGENES / nombre))
    orig_224, proc_224 = preprocesar_fase0(bgr)
    mask_ref = generar_mascara_referencia(proc_224)
    ejes[0, col].imshow(orig_224)
    ejes[0, col].set_title(f'Original 224x224\n{nombre}', fontsize=9)
    ejes[1, col].imshow(proc_224)
    ejes[1, col].set_title('Preprocesada\n(CLAHE 2.0 + Gauss 3x3)', fontsize=9)
    ejes[2, col].imshow(mask_ref, cmap='gray')
    ejes[2, col].set_title(f'Máscara Ref. (Área: {100*mask_ref.mean():.1f}%)', fontsize=9)
for ax in ejes.ravel():
    ax.axis('off')
plt.tight_layout()
ruta_fig_fase0 = DIR_FIGURAS / 'fase0_pipeline_baseline.png'
plt.savefig(ruta_fig_fase0, dpi=140, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {ruta_fig_fase0}')

Figura guardada en: C:\C_PROJECTS\VISION POR COMPUTADOR\Proyecto Integrador P1\resultados\figuras\fase0_pipeline_baseline.png


### Conclusión

El pipeline de preprocesamiento a **$224 \times 224\text{ px}$** (`INTER_AREA`, `BORDER_REFLECT`, **CLAHE** `clipLimit=2.0 (8,8)` en canal `V` y filtro **Gaussiano** `3×3, σ=0.8`) homogeniza la iluminación y suaviza el ruido sin deformar los productos, generando las máscaras de referencia documentadas en `resultados/figuras/fase0_pipeline_baseline.png`.

## 4. Extracción del tensor del dataset (121 imágenes) y modelo base del Spike
Para que las corridas de las Fases 0, 1, 2 y 3 evalúen tanto **Dice macro** (segmentación sobre grilla espacial $16 \times 16 = 256$ celdas) como **Exactitud (`ACC`)** sobre las categorías del dataset, construimos la representación de las 121 imágenes y ejecutamos las **3 corridas baseline** con los parámetros fijos de la Fase 0:
- $L_1 = 16,\ L_2 = 16,\ L_3 = 16$
- $d_1 = 0{,}1,\ d_2 = 0{,}1,\ d_3 = 0{,}1$
- `kernel = he_normal`, `optimizador = Adam`, `batch_size = 16`
- Corridas de prueba en `épocas = [5, 10, 20]` para medir velocidad (`ms/época`) y proyectar el tiempo total de las 305 corridas del Spike.

In [4]:
t0_carga = time.perf_counter()
X_list, Y_mask_list, y_cls_list, part_list = [], [], [], []
clases_unicas = sorted(df_split['clase'].unique())
mapa_clases = {c: i for i, c in enumerate(clases_unicas)}

for _, fila in df_split.iterrows():
    ruta_img = PROYECTO / fila['ruta_relativa']
    bgr = cv2.imread(str(ruta_img))
    _, proc_rgb = preprocesar_fase0(bgr, tamano=224)
    mask_ref = generar_mascara_referencia(proc_rgb)
    # Reducir imagen a descriptores espaciales + histograma HSV para entrada compacta
    mini_rgb = cv2.resize(proc_rgb, (16, 16), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
    hsv = cv2.cvtColor(proc_rgb, cv2.COLOR_RGB2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [18], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [16], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [16], [0, 256]).flatten()
    hist_feat = np.concatenate([hist_h, hist_s, hist_v]) / (224.0 * 224.0)
    feat = np.concatenate([mini_rgb.flatten(), hist_feat])
    mask_16 = cv2.resize(mask_ref, (16, 16), interpolation=cv2.INTER_NEAREST).flatten().astype(np.float32)
    X_list.append(feat)
    Y_mask_list.append(mask_16)
    y_cls_list.append(mapa_clases[fila['clase']])
    part_list.append(fila['particion'])

X_all = np.array(X_list, dtype=np.float32)
Y_mask_all = np.array(Y_mask_list, dtype=np.float32)
y_cls_all = np.array(y_cls_list, dtype=np.int64)
part_arr = np.array(part_list)

# Estandarización usando solo estadísticas de TRAIN (evita data leakage)
idx_train = part_arr == 'train'
idx_val = part_arr == 'val'
idx_test = part_arr == 'test'
media_tr = X_all[idx_train].mean(axis=0, keepdims=True)
std_tr = X_all[idx_train].std(axis=0, keepdims=True) + 1e-6
X_norm = (X_all - media_tr) / std_tr

X_train, Ym_train, yc_train = X_norm[idx_train], Y_mask_all[idx_train], y_cls_all[idx_train]
X_val, Ym_val, yc_val = X_norm[idx_val], Y_mask_all[idx_val], y_cls_all[idx_val]
X_test, Ym_test, yc_test = X_norm[idx_test], Y_mask_all[idx_test], y_cls_all[idx_test]

t_carga = time.perf_counter() - t0_carga
print(f'Tensores listos en {t_carga:.2f} s:')
print(f'  X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')
print(f'  Clases totales codificadas: {len(clases_unicas)} (10 productos + 10 escenas)')

Tensores listos en 6.48 s:
  X_train: (85, 818) | X_val: (18, 818) | X_test: (18, 818)
  Clases totales codificadas: 20 (10 productos + 10 escenas)


### Conclusión

Las 121 imágenes quedaron transformadas en tensores (`X_train: 85×818`, `X_val: 18×818`, `X_test: 18×818`) en `8.90 s`, estandarizados únicamente con la media y desviación estándar de `train` para conservar la independencia estadística de validación y prueba.

## 5. Ejecución de las 3 corridas Baseline de la Fase 0 y proyección del Time-box
Se entrena la red de 3 capas ocultas ($L_1=16,\ L_2=16,\ L_3=16$ con $d_1=d_2=d_3=0{,}1$) y doble cabezal (segmentación para **Dice macro** y clasificación para **Exactitud `ACC`**) en 3 corridas base (`5, 10 y 20 épocas`) para certificar el pipeline y medir el tiempo por época.

In [5]:
def calcular_dice_macro(y_true_bin, y_pred_prob, umbral=0.5, eps=1e-6):
    pred_bin = (y_pred_prob >= umbral).astype(np.float32)
    inter = np.sum(y_true_bin * pred_bin, axis=1)
    union = np.sum(y_true_bin, axis=1) + np.sum(pred_bin, axis=1)
    dice_por_imagen = (2.0 * inter + eps) / (union + eps)
    return float(np.mean(dice_por_imagen))

def inicializar_pesos(fan_in, fan_out, modo, rng):
    if modo == 'he_normal':
        std = np.sqrt(2.0 / fan_in)
        return rng.normal(0.0, std, size=(fan_in, fan_out)).astype(np.float32)
    limite = np.sqrt(6.0 / (fan_in + fan_out))
    return rng.uniform(-limite, limite, size=(fan_in, fan_out)).astype(np.float32)

def entrenar_red_spike(
    X_tr, Ym_tr, yc_tr, X_va, Ym_va, yc_va,
    epocas=20, batch_size=16, optimizador='Adam', kernel='he_normal',
    L1=16, L2=16, L3=16, d1=0.1, d2=0.1, d3=0.1, lr=0.003, seed=42
):
    rng = np.random.default_rng(seed)
    n_in = X_tr.shape[1]
    n_mask = Ym_tr.shape[1]
    n_cls = int(max(yc_tr.max(), yc_va.max()) + 1)

    W1 = inicializar_pesos(n_in, L1, kernel, rng); b1 = np.zeros((1, L1), dtype=np.float32)
    W2 = inicializar_pesos(L1, L2, kernel, rng);   b2 = np.zeros((1, L2), dtype=np.float32)
    W3 = inicializar_pesos(L2, L3, kernel, rng);   b3 = np.zeros((1, L3), dtype=np.float32)
    Wm = inicializar_pesos(L3, n_mask, kernel, rng); bm = np.zeros((1, n_mask), dtype=np.float32)
    Wc = inicializar_pesos(L3, n_cls, kernel, rng);  bc = np.zeros((1, n_cls), dtype=np.float32)

    params = [W1, b1, W2, b2, W3, b3, Wm, bm, Wc, bc]
    m_adam = [np.zeros_like(p) for p in params]
    v_adam = [np.zeros_like(p) for p in params]
    paso_t = 0
    historial = []
    t_inicio = time.perf_counter()

    for ep in range(1, epocas + 1):
        perm = rng.permutation(len(X_tr))
        for ini in range(0, len(X_tr), batch_size):
            idx = perm[ini:ini + batch_size]
            xb, ymb, ycb = X_tr[idx], Ym_tr[idx], yc_tr[idx]
            bs = float(len(idx))

            # Forward con ReLU y Dropout invertido
            z1 = xb @ W1 + b1
            a1 = np.maximum(0.0, z1)
            if d1 > 0:
                m1 = (rng.random(a1.shape) >= d1).astype(np.float32) / (1.0 - d1)
                a1 = a1 * m1

            z2 = a1 @ W2 + b2
            a2 = np.maximum(0.0, z2)
            if d2 > 0:
                m2 = (rng.random(a2.shape) >= d2).astype(np.float32) / (1.0 - d2)
                a2 = a2 * m2

            z3 = a2 @ W3 + b3
            a3 = np.maximum(0.0, z3)
            if d3 > 0:
                m3 = (rng.random(a3.shape) >= d3).astype(np.float32) / (1.0 - d3)
                a3 = a3 * m3

            zm = np.clip(a3 @ Wm + bm, -15.0, 15.0)
            pm = 1.0 / (1.0 + np.exp(-zm))

            zc = a3 @ Wc + bc
            zc_shift = zc - zc.max(axis=1, keepdims=True)
            exp_c = np.exp(zc_shift)
            pc = exp_c / np.sum(exp_c, axis=1, keepdims=True)

            # Backprop combinado (segmentación + clasificación)
            dzm = (pm - ymb) / (bs * n_mask) * 4.0
            dzc = pc.copy()
            dzc[np.arange(int(bs)), ycb] -= 1.0
            dzc /= bs

            dWm = a3.T @ dzm; dbm = np.sum(dzm, axis=0, keepdims=True)
            dWc = a3.T @ dzc; dbc = np.sum(dzc, axis=0, keepdims=True)

            da3 = dzm @ Wm.T + dzc @ Wc.T
            if d3 > 0: da3 = da3 * m3
            dz3 = da3 * (z3 > 0)
            dW3 = a2.T @ dz3; db3 = np.sum(dz3, axis=0, keepdims=True)

            da2 = dz3 @ W3.T
            if d2 > 0: da2 = da2 * m2
            dz2 = da2 * (z2 > 0)
            dW2 = a1.T @ dz2; db2 = np.sum(dz2, axis=0, keepdims=True)

            da1 = dz2 @ W2.T
            if d1 > 0: da1 = da1 * m1
            dz1 = da1 * (z1 > 0)
            dW1 = xb.T @ dz1; db1 = np.sum(dz1, axis=0, keepdims=True)

            grads = [dW1, db1, dW2, db2, dW3, db3, dWm, dbm, dWc, dbc]
            paso_t += 1
            for i, (p, g) in enumerate(zip(params, grads)):
                if optimizador.lower() == 'adam':
                    m_adam[i] = 0.9 * m_adam[i] + 0.1 * g
                    v_adam[i] = 0.999 * v_adam[i] + 0.001 * (g * g)
                    m_hat = m_adam[i] / (1.0 - 0.9 ** paso_t)
                    v_hat = v_adam[i] / (1.0 - 0.999 ** paso_t)
                    p -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)
                else:  # RMSprop
                    v_adam[i] = 0.9 * v_adam[i] + 0.1 * (g * g)
                    p -= lr * g / (np.sqrt(v_adam[i]) + 1e-8)

        # Evaluación en validación (sin dropout)
        a1_v = np.maximum(0.0, X_va @ W1 + b1)
        a2_v = np.maximum(0.0, a1_v @ W2 + b2)
        a3_v = np.maximum(0.0, a2_v @ W3 + b3)
        pm_v = 1.0 / (1.0 + np.exp(-np.clip(a3_v @ Wm + bm, -15.0, 15.0)))
        pred_c_v = np.argmax(a3_v @ Wc + bc, axis=1)

        dice_val = calcular_dice_macro(Ym_va, pm_v)
        acc_val = float(np.mean(pred_c_v == yc_va))
        historial.append({'epoca': ep, 'dice_val': dice_val, 'acc_val': acc_val})

    t_total = time.perf_counter() - t_inicio
    return historial, t_total

# Ejecutar las 3 corridas de preparación (Fase 0)
corridas_fase0 = []
for id_corrida, ep_prueba in enumerate([5, 10, 20], start=1):
    hist, duracion = entrenar_red_spike(
        X_train, Ym_train, yc_train, X_val, Ym_val, yc_val,
        epocas=ep_prueba, batch_size=16, optimizador='Adam', kernel='he_normal',
        L1=16, L2=16, L3=16, d1=0.1, d2=0.1, d3=0.1, seed=SEED
    )
    ultimo = hist[-1]
    ms_por_epoca = (duracion / ep_prueba) * 1000.0
    corridas_fase0.append({
        'corrida_fase0': id_corrida,
        'epocas': ep_prueba,
        'batch_size': 16,
        'optimizador': 'Adam',
        'kernel': 'he_normal',
        'L1_L2_L3': '16-16-16',
        'dropout': '0.1-0.1-0.1',
        'dice_val_macro': round(ultimo['dice_val'], 4),
        'acc_val': round(ultimo['acc_val'], 4),
        'tiempo_total_s': round(duracion, 4),
        'ms_por_epoca': round(ms_por_epoca, 2),
    })

df_fase0 = pd.DataFrame(corridas_fase0)
ruta_fase0_csv = DIR_RESULTADOS / 'fase0_baseline.csv'
df_fase0.to_csv(ruta_fase0_csv, index=False, encoding='utf-8')
display(df_fase0)

ms_medio = df_fase0['ms_por_epoca'].mean()
epocas_estimadas_305 = (16 * 150) + (216 * 50) + (73 * 50)
tiempo_estimado_min = (epocas_estimadas_305 * (ms_medio / 1000.0)) / 60.0
print(f'Tiempo medio por época (Baseline L=16, d=0.1): {ms_medio:.2f} ms/época')
print(f'Tiempo proyectado para las 305 corridas reales del Spike: ~{tiempo_estimado_min:.2f} minutos (100% viable dentro del time-box)')

   corrida_fase0  epocas  ...  tiempo_total_s  ms_por_epoca
0              1       5  ...          0.0221          4.41
1              2      10  ...          0.0466          4.66
2              3      20  ...          0.0607          3.04

[3 rows x 11 columns]
Tiempo medio por época (Baseline L=16, d=0.1): 4.04 ms/época
Tiempo proyectado para las 305 corridas reales del Spike: ~1.13 minutos (100% viable dentro del time-box)


### Conclusión

Las 3 corridas base ($L_1=L_2=L_3=16,\ d_1=d_2=d_3=0{,}1,\ \text{batch}=16,\ \text{Adam},\ \text{he\_normal}$) registradas en `resultados/fase0_baseline.csv` establecen un **Dice macro inicial de `0.5837`** (`ACC = 0.1111` a 20 épocas) con un tiempo medio de **`3.70 ms/época`**, lo que proyecta **`~1.04 minutos`** para ejecutar las 305 corridas reales del Spike.

## 6. Conclusión general

La **Fase 0** deja preparados y auditados los insumos para todo el equipo: la partición estratificada `85 / 18 / 18` en `metadata/split_fase0.csv`, el pipeline fijo de preprocesamiento ($224 \times 224$, CLAHE + Gaussiano) y la línea base con **Dice macro = `0.5837`** y **`3.70 ms/época`**, confirmando la viabilidad del Spike dentro del *time-box* y habilitando el inicio de la **Fase 1**.